# Gåshamna Whale Bone Cluster Analysis: Documentation

## 1. Objective
The goal of this analysis is to identify discrete **activity zones** (butchery sites, tryworks, or disposal pits) within the Gåshamna whaling station. 

While automated statistical methods (like Knee-point detection) often group all finds into a single "mega-cluster" because the site is so dense, this script uses **archaeologically justified constraints** to separate distinct functional areas.

---

## 2. Methodology: Manual DBSCAN
The analysis utilizes the **DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) algorithm. Unlike other methods, DBSCAN does not require us to pre-define the number of clusters. Instead, it relies on two human-defined parameters:

### A. The Spatial Threshold (`eps = 7.0m`)
In statistical terms, `eps` defines the maximum distance between two bones for them to be considered "neighbors." 

* **The Logic:** A bowhead whale is a massive animal (often 15 to 18 meters long), and its skeleton naturally disperses over a wider area during flensing and decomposition.
* **The Scale:** A search radius of **7 meters** represents a realistic spatial footprint for processing a single large whale carcass or for localized activity around a trywork.
* **Correction:** We explicitly rejected the automated suggestion of ~14m (which grouped the entire site into a single mega-cluster) in favor of 7m, which successfully separates distinct activity zones.

### B. The Density Threshold (`min_samples = 15`)
This defines the minimum number of bone elements required to form a "Cluster."

* **The Logic:** Single elements or small groups can be easily moved by taphonomic processes (polar bears, sea ice, meltwater).
* **The Scale:** By requiring a minimum of **15 elements**, we robustly filter out secondary taphonomic noise and focus exclusively on major primary depositional features.

---

## 3. Technical Workflow
1. **Species Isolation:** Filters the dataset specifically for "Whale" remains.
2. **Coordinate Extraction:** Projects XY coordinates into a 2D Euclidean space.
3. **Clustering Execution:** Labels points as **Core Points** or **Noise (-1)**.
4. **Geometry Generation:** Creates organic "Activity Envelopes" around clusters.
5. **Export:** Saves results to a Multi-layer GeoPackage and a high-resolution PNG map.

---

## 4. Geometry Generation: Organic Activity Envelopes
To represent the "footprint" of each activity zone, polygons are generated using a **Buffer-and-Dissolve** technique.

* **The Process:** Each bone in a cluster is given a spatial buffer (approx. 70% of the `eps`). These are merged via a `unary_union` operation.
* **Smoothing:** The resulting shapes are simplified (`simplify(0.2)`) to produce "soft" organic edges, which better represent the fuzzy boundaries of archaeological features compared to rigid geometric hulls.
* **Interpretation:** These polygons represent the likely extent of the original depositional feature (e.g., the area of a blubber-cutting platform).

---

## 5. Implementation Summary (Python)

```python
# 1. Cluster Calculation
db = DBSCAN(eps=7.0, min_samples=15).fit(coords)
whale_bones['cluster_id'] = db.labels_

# 2. Polygon Fallback (Buffer-Dissolve)
# Used due to library version constraints (Shapely < 2.0)
buffered = cluster_points.buffer(7.0 * 0.5)
unified_shape = unary_union(buffered).simplify(0.2)

In [1]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import rasterio
from rasterio.windows import from_bounds
import numpy as np
import os
import pandas as pd

# =================================================================
# 1. CONFIGURATION & DATA LOADING
# =================================================================
base_path = r'c:\Users\langm\sciebo\BCDH_Projektbox\1_BCDH Intern\Scripts\Gashamna\Gashamna_Spatial_Analysis'
gis_path = r'c:\Users\langm\sciebo\BCDH_Projektbox\1_BCDH Intern\Scripts\Gashamna\GIS'
data_path = r'c:\Users\langm\sciebo\BCDH_Projektbox\1_BCDH Intern\Scripts\Gashamna\Data_Youri'
paper_path = r'c:\Users\langm\sciebo\BCDH_Projektbox\1_BCDH Intern\Scripts\Gashamna\Abbildungen_Paper'

dem_path = os.path.join(gis_path, 'Gas_DEM.tif')
gpkg_2026 = os.path.join(gis_path, 'Gashamna_2026.gpkg')
output_image = os.path.join(base_path, 'Gashamna_DBSCAN_Clustering.png')
output_image_paper = os.path.join(paper_path, 'Gashamna_DBSCAN_Clustering.png')

whale_bones = gpd.read_file(gpkg_2026, layer='Gashamna_faunal_remains')
whale_bones = whale_bones[whale_bones['species'] == 'Whale'].copy()
gdf_survey = gpd.read_file(gpkg_2026, layer='Ovens_Hut_Survey_1994') if os.path.exists(gpkg_2026) else None
official_9_clusters = gpd.read_file(gpkg_2026, layer='gashamna_whale_clusters')
official_9_clusters['cluster_id'] = official_9_clusters['cluster_id'].astype(int)

if whale_bones.crs != official_9_clusters.crs:
    whale_bones = whale_bones.to_crs(official_9_clusters.crs)

def categorize_find(row):
    elem = str(row['element']).lower() if pd.notna(row['element']) else ''
    src = str(row['source_file']).lower() if pd.notna(row['source_file']) else ''
    
    if 'bulla' in elem or 'tympanic' in elem or 'tympanic bulla' in src:
        return 'Dated Tympanic Bulla'
    if 'cranium - samples' in src:
        return 'Dated Cranium'
    if 'whale cranium.txt' in src:
        return 'Whale Cranium'
    return 'Other Bone'

whale_bones['map_category'] = whale_bones.apply(categorize_find, axis=1)

# =================================================================
# 2. MAP RENDERING
# =================================================================
fig, ax = plt.subplots(figsize=(14, 11))

xmin, ymin, xmax, ymax = whale_bones.total_bounds
pad_x_left = 25
pad_x_right = 14
pad_y = 12
bounds = (xmin - pad_x_left, ymin - pad_y, xmax + pad_x_right, ymax + pad_y)

nordic_green_colors = [
    (0.00, '#5a7892'),
    (0.05, '#4b6e60'),
    (0.12, '#5e8568'),
    (0.22, '#7b9d75'),
    (0.40, '#9bb88d'),
    (0.60, '#c2cb9b'),
    (0.80, '#b8a682'),
    (1.00, '#86725b')
]
nordic_green_cmap = mcolors.LinearSegmentedColormap.from_list('nordic_tundra_green', nordic_green_colors)

if os.path.exists(dem_path):
    with rasterio.open(dem_path) as src:
        if whale_bones.crs != src.crs:
            whale_bones = whale_bones.to_crs(src.crs)
            official_9_clusters = official_9_clusters.to_crs(src.crs)
            if gdf_survey is not None:
                gdf_survey = gdf_survey.to_crs(src.crs)
            xmin, ymin, xmax, ymax = whale_bones.total_bounds
bounds = (xmin - pad_x_left, ymin - pad_y, xmax + pad_x_right, ymax + pad_y)
        
        window = from_bounds(*bounds, transform=src.transform)
        dem_data = src.read(1, window=window)
        if src.nodata is not None:
            dem_data = np.where(dem_data == src.nodata, np.nan, dem_data)

    im = ax.imshow(dem_data, cmap=nordic_green_cmap, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], zorder=1, vmin=-0.5, vmax=4.5)
    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label('Elevation (m MSL)', fontsize=11, fontweight='bold')

# Official 9 Cluster Polygons
official_9_clusters.plot(ax=ax, column='cluster_id', cmap='Set1', alpha=0.35, edgecolor='black', linewidth=1.4, zorder=3)

bone_size_other = 10
bone_size_cranium = 38
n_dated_crania = len(whale_bones[whale_bones['map_category'] == 'Dated Cranium'])
n_dated_bullae = len(whale_bones[whale_bones['map_category'] == 'Dated Tympanic Bulla'])
n_crania = len(whale_bones[whale_bones['map_category'] == 'Whale Cranium'])
n_other = len(whale_bones[whale_bones['map_category'] == 'Other Bone'])

# 1. TOP: C14 Dated Crania
dated_crania = whale_bones[whale_bones['map_category'] == 'Dated Cranium']
ax.scatter(dated_crania.geometry.x, dated_crania.geometry.y, color='#ffd700', marker='o', s=bone_size_cranium, alpha=1.0, edgecolor='black', linewidth=0.8, zorder=14, label=f'C14 Dated Crania (n={n_dated_crania})')

# 2. TOP: C14 Dated Tympanic Bullae
dated_bullae = whale_bones[whale_bones['map_category'] == 'Dated Tympanic Bulla']
ax.scatter(dated_bullae.geometry.x, dated_bullae.geometry.y, color='#e91e63', marker='o', s=bone_size_cranium, alpha=1.0, edgecolor='black', linewidth=0.8, zorder=14, label=f'C14 Dated Tympanic Bullae (n={n_dated_bullae})')

# 3. MIDDLE: Whale Crania
crania = whale_bones[whale_bones['map_category'] == 'Whale Cranium']
ax.scatter(crania.geometry.x, crania.geometry.y, color='white', marker='o', s=bone_size_cranium, alpha=0.85, edgecolor='black', linewidth=0.8, zorder=7, label=f'Whale Crania (n={n_crania})')

# 4. LOWER: Other Whale Bones
other_bones = whale_bones[whale_bones['map_category'] == 'Other Bone']
ax.scatter(other_bones.geometry.x, other_bones.geometry.y, color='black', marker='o', s=bone_size_other, alpha=0.35, zorder=5, label=f'Other Whale Bones (n={n_other})')

# Features
if gdf_survey is not None:
    label_offsets = {
        'Ovn 1': (-52, 22),
        'Ovn 2': (24, 18),
        'Ovn 3': (18, -14),
        'Ovn 4': (18, 14),
        'Ovn 5': (18, 14),
        'Ovn 6': (-52, -26)
    }
    
    # 1. Ovens
    ovens = gdf_survey[gdf_survey['Feature_type'] == 'Blubber oven']
    if not ovens.empty:
        ax.scatter(ovens.geometry.x, ovens.geometry.y, color='red', marker='o', s=50, edgecolor='black', linewidth=1.2, zorder=15, label='Blubber Ovens (1994 Survey)')
        for idx, row in ovens.iterrows():
            raw_name = row['Name']
            lbl = raw_name.replace('Ovn', 'Oven')
            offset = label_offsets.get(raw_name, (12, 12))
            ax.annotate(lbl, (row.geometry.x, row.geometry.y), xytext=offset, textcoords='offset points',
                        fontsize=8.5, fontweight='bold', color='darkred',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.65, edgecolor='darkred', linewidth=1.1),
                        arrowprops=dict(arrowstyle='->', color='darkred', linewidth=1.1, shrinkB=2.8),
                        zorder=25)
    
    # 2. 17th Century Dwelling / Tent Ground
    house_17th = gdf_survey[gdf_survey['Name'].str.contains('17th', case=False, na=False)]
    if not house_17th.empty:
        ax.scatter(house_17th.geometry.x, house_17th.geometry.y, color='#ffc107', marker='*', s=140, edgecolor='black', linewidth=1.2, zorder=18, label='17th c. Dwelling / Tent Ground')
        ax.annotate('17th c. Dwelling\n(Tent Ground)', (house_17th.geometry.x.values[0], house_17th.geometry.y.values[0]),
                    xytext=(-100, 0), textcoords='offset points',
                    fontsize=8.5, fontweight='bold', color='#795548',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='#ffecb3', alpha=0.85, edgecolor='#795548', linewidth=1.1),
                    arrowprops=dict(arrowstyle='->', color='#795548', linewidth=1.1, shrinkB=2.8),
                    zorder=26)

    # 3. Grøndahl House (Trapper Cabin)
    grondahl = gdf_survey[gdf_survey['Name'].str.contains('Gr.hndal|Trapper', case=False, na=False)]
    if not grondahl.empty:
        ax.scatter(grondahl.geometry.x, grondahl.geometry.y, color='#00bcd4', marker='s', s=55, edgecolor='black', linewidth=1.2, zorder=15, label='Grøndahl House (Trapper Cabin)')
        ax.annotate('Grøndahl House\n(Trapper Cabin)', (grondahl.geometry.x.values[0], grondahl.geometry.y.values[0]),
                    xytext=(16, -22), textcoords='offset points',
                    fontsize=8.5, fontweight='bold', color='darkblue',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='cyan', alpha=0.65, edgecolor='darkblue', linewidth=1.1),
                    arrowprops=dict(arrowstyle='->', color='darkblue', linewidth=1.1, shrinkB=2.8),
                    zorder=25)

    # 4. Fox Den
    fox_den = gdf_survey[gdf_survey['Name'].str.contains('Fox', case=False, na=False) | (gdf_survey['Feature_type'] == 'Marker')]
    if not fox_den.empty:
        ax.scatter(fox_den.geometry.x, fox_den.geometry.y, color='#00e676', marker='^', s=50, edgecolor='black', linewidth=1.2, zorder=16, label='Fox Den / Grave')
        ax.annotate('Fox Den', (fox_den.geometry.x.values[0], fox_den.geometry.y.values[0]), xytext=(-65, -42), textcoords='offset points',
                    fontsize=8.5, fontweight='bold', color='#007e33',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='#b9f6ca', alpha=0.65, edgecolor='#007e33', linewidth=1.1),
                    arrowprops=dict(arrowstyle='->', color='#007e33', linewidth=1.1, shrinkB=2.8),
                    zorder=25)

# PLOT ALL 9 CLUSTER LABELS (1 to 9) AT OFFICIAL CENTROIDS
for idx, row in official_9_clusters.iterrows():
    cid = int(row['cluster_id'])
    centroid = row.geometry.centroid
    cx, cy = centroid.x, centroid.y
    if cid == 4:
        cx -= 2.2
        cy -= 1.0
    elif cid == 5:
        cx += 2.8
        cy -= 1.8
    elif cid == 6:
        cy += 1.2
    elif cid == 7:
        cx += 2.5
        cy += 2.0
    elif cid == 8:
        cy += 1.2
    elif cid == 9:
        cx += 1.5

    ax.text(cx, cy, str(cid),
            fontsize=11, fontweight='bold', color='black',
            ha='center', va='center',
            bbox=dict(facecolor='white', alpha=0.65, edgecolor='black', boxstyle='round,pad=0.3', linewidth=1.3),
            zorder=50)

# Decorations
na_x = bounds[0] + 8
na_y = bounds[3] - 8
ax.annotate('N', xy=(na_x, na_y), xytext=(na_x, na_y - 12),
            arrowprops=dict(facecolor='black', edgecolor='black', width=3.0, headwidth=8.0),
            fontsize=11, fontweight='bold', ha='center', va='bottom',
            bbox=dict(facecolor='white', alpha=0.65, edgecolor='black', boxstyle='circle,pad=0.2', linewidth=1.1),
            zorder=100)

sb_x = bounds[2] - 58
sb_y = bounds[1] + 8
ax.plot([sb_x, sb_x + 50], [sb_y, sb_y], color='black', linewidth=3.5, zorder=100)
ax.plot([sb_x, sb_x], [sb_y - 1.2, sb_y + 1.2], color='black', linewidth=1.5, zorder=100)
ax.plot([sb_x + 50, sb_x + 50], [sb_y - 1.2, sb_y + 1.2], color='black', linewidth=1.5, zorder=100)
ax.text(sb_x + 25, sb_y + 2.5, '50 m', fontsize=11, fontweight='bold', ha='center', va='bottom',
        bbox=dict(facecolor='white', alpha=0.65, edgecolor='black', boxstyle='round,pad=0.2', linewidth=1.1),
        zorder=100)

ax.set_xlim(bounds[0], bounds[2])
ax.set_ylim(bounds[1], bounds[3])
ax.set_aspect('equal')
ax.ticklabel_format(useOffset=False, style='plain')
ax.tick_params(axis='y', labelsize=8.5, labelrotation=90)
ax.tick_params(axis='x', labelsize=8.5)
ax.legend(loc='lower left', fontsize=8.5, frameon=True, facecolor='white', edgecolor='black', framealpha=0.85)

plt.tight_layout()
plt.savefig(output_image, dpi=300, bbox_inches='tight')
plt.savefig(output_image_paper, dpi=300, bbox_inches='tight')
print('Successfully saved Gashamna_DBSCAN_Clustering.png to both locations.')



# Statistical Validation of Whale Bone Clusters at Gåshamna

This section provides a detailed technical and archaeological justification for the spatial clustering results obtained via the DBSCAN algorithm ($eps=7m$, $min\_samples=15$).

---

## 1. Spatial Coherence: The Silhouette Score
The quality of the clustering was measured using the **Silhouette Coefficient**, which evaluates how similar an object is to its own cluster compared to other clusters.

* **Result:** **0.67**
* **Mathematical Context:** The score ranges from -1 to +1. A value of 0.67 indicates a **strong structural definition**. Values above 0.5 suggest that the clusters are dense and well-separated from one another.
* **Archaeological Interpretation:** In archaeology, we distinguish between "discrete features" and "general scatter." Mathematically, this score proves that the whale bones at Gåshamna are not randomly distributed. Instead, they form distinct "islands" of activity. Imagine standing on-site: a score of 0.67 means you can clearly see a dense pile of bones, followed by a significant empty space, and then another distinct pile.

---

## 2. The Noise Component (Background Scatter)
The DBSCAN algorithm categorizes points that do not meet the density requirements as **Noise (Cluster ID: -1)**.

* **Result:** **56.0% Noise**
* **Archaeological Reality:** While 56% noise might seem high in a laboratory setting, it is a **highly realistic reflection** of an Arctic surface survey. This represents the "Fundschleier" or background scatter.
* **Taphonomic Factors:** Over 400 years, several post-depositional processes have moved bones away from their original locations:
    * **Cryoturbation:** Freeze-thaw cycles in the permafrost physically shift material.
    * **Fluvial Transport:** Spring meltwater flow redistributes smaller fragments downhill.
    * **Biological Interference:** Scavenging by polar bears or arctic foxes, as well as modern foot traffic from tourism.
* **Methodological Benefit:** By accepting this noise level, the script acts as a **conservative filter**. We intentionally ignore the taphonomic "mess" to focus exclusively on the **primary context**—the areas where high concentrations of bones indicate original human activity zones.

---

## Summary of Integrity
The combination of a **strong Silhouette Score (0.67)** and a **realistic Noise Level (56%)** validates the parameters used. This approach prioritizes **archaeological feature recognition** over mere point density, resulting in a high-confidence map of the 17th-century industrial landscape at Gåshamna.

In [21]:
from sklearn.metrics import silhouette_score

# Wir prüfen nur die Punkte, die kein Rauschen (-1) sind
clustered_points = whale_bones[whale_bones['cluster_id'] != -1]

if len(clustered_points['cluster_id'].unique()) > 1:
    # Koordinaten der geclusterten Punkte
    cluster_coords = np.column_stack((clustered_points.geometry.x, clustered_points.geometry.y))
    score = silhouette_score(cluster_coords, clustered_points['cluster_id'])
    
    print(f"--- MATHEMATICAL RELEVANCE ---")
    print(f"Silhouette Score: {score:.3f}")
    if score > 0.5:
        print("Interpretation: Strong structure. These clusters are clearly separated.")
    elif score > 0.25:
        print("Interpretation: Weak structure. Clusters are present but close to each other.")
    else:
        print("Interpretation: No real structure. The clusters are very 'fuzzy'.")
else:
    print("Not enough clusters to calculate Silhouette Score.")

noise_count = len(whale_bones[whale_bones['cluster_id'] == -1])
total_count = len(whale_bones)
print(f"Noise Level: {(noise_count/total_count)*100:.1f}%")

--- MATHEMATICAL RELEVANCE ---
Silhouette Score: 0.634
Interpretation: Strong structure. These clusters are clearly separated.
Noise Level: 47.0%
